In [0]:
%sql
USE CATALOG laddstolpar_df;

CREATE SCHEMA IF NOT EXISTS landing COMMENT 'Raw API responses as files, unchanged';
CREATE SCHEMA IF NOT EXISTS ops     COMMENT 'Run log and validation results';

CREATE VOLUME IF NOT EXISTS landing.raw
  COMMENT 'One file per API call, deterministic paths: <source>/<partition>/<file>';

DESCRIBE VOLUME landing.raw;

In [0]:
import requests, json
from datetime import date

BASE = "https://www.elprisetjustnu.se/api/v1/prices"

def price_url(d: date, zone: str) -> str:
    return f"{BASE}/{d:%Y}/{d:%m-%d}_{zone}.json"

# (date, what we expect to see)
tests = [
    (date(2025, 9, 30),  "last hourly day   -> 24 rows"),
    (date(2025, 10, 1),  "first 15-min day  -> 96 rows"),
    (date(2025, 10, 26), "DST ends (25 h)   -> 100 rows"),
    (date(2026, 9, 23),  "normal recent day -> 96 rows"),
]

for d, expect in tests:
    r = requests.get(price_url(d, "SE3"), timeout=30)
    rows = r.json() if r.status_code == 200 else []
    print(f"{d}  HTTP {r.status_code}  rows={len(rows):>3}  expected: {expect}")

# Field names and one example row
print(json.dumps(rows[:1], indent=2))